# Conference study: measured TP motion, POD representations, and quantum kernels

This notebook builds the computational evidence for five linked questions:

1. How repeatable are TP measurements, and how well do automatic tracks agree with blind manual repeats?
2. How well do POD features represent continuous TP motion?
3. Which **fixed-dimension** feature/crop representation works best on validation?
4. How do RBF-SVM and fidelity-kernel SVM compare as classifier sample count grows?
5. Where do boundary errors occur, and what are the measured classical/simulator costs?

Reuse completed, reviewed outputs from `Triple_Point_Classification.ipynb`; no initial-20 retracking is required. This study does not create a measurement uncertainty from physical oscillation SD. It needs new blind repeat clicks. It supports a conference analysis, but experimental validity and paper completeness remain scientific judgments.

**Study discipline:** training/validation select the representation and hyperparameters; a saved recipe freezes them before test evaluation. The already inspected test block is labeled retrospective by default. A fresh final claim needs an actually unseen holdout or record.

## 1. Parameters and output locations

Keep this notebook beside the updated `schlieren_qml` folder. Install requirements into the active Anaconda kernel if needed and restart after updating code. Use a new `STUDY_TAG` when changing a completed study; outputs are protected from replacement. Stage switches are initially off so Run All cannot launch windows or expensive studies unexpectedly.

Your existing saved experiment determines the original path, crop, acquisition timing and split. The default comparison crop extends leftward; inspect it against the complete TP excursion before enabling the study.

In [ ]:
%matplotlib inline
from pathlib import Path
from datetime import datetime
import sys, json
from IPython.display import display, Image

PROJECT_DIR = Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from schlieren_qml.paper_data import load_study, write_json, rows
from schlieren_qml.paper_measurement import (
    prepare_repeat_sample, launch_repeat_annotation, measurement_report,
)
from schlieren_qml.paper_study import run_validation_study, evaluate_frozen_test

TP_DIRECTORY = PROJECT_DIR / "results-TP" / "tp_measurement"
RESULTS_DIRECTORY = TP_DIRECTORY / "categories_pod_qsvm"
STATES_CSV = TP_DIRECTORY / "categories" / "tp_states.csv"
STUDY_TAG = "conference_v1"
PAPER_DIRECTORY = PROJECT_DIR / "results-TP" / STUDY_TAG
REPEAT_DIRECTORY = PAPER_DIRECTORY / "repeat_annotations"
MEASUREMENT_REPORT_ROOT = PAPER_DIRECTORY / "measurement_reports"
VALIDATION_DIRECTORY = PAPER_DIRECTORY / "validation_study"
FINAL_DIRECTORY = PAPER_DIRECTORY / "frozen_test"

DESKTOP_BACKEND = "TkAgg"
N_REPEAT_TRAIN = 60
N_REPEAT_VALIDATION = 30
REQUIRE_MEASURED_UNCERTAINTY = True
ASSUMED_SENSITIVITY_MARGIN_PX = 2.0  # Used only if exploratory runs permit no measured uncertainty.

FEATURE_DIMENSION = 4  # Same dimension/qubits for every classifier comparison.
CROP_STUDIES = {"wider_interaction": [320, 140, 750, 510]}
# Use {} to compare leading/ranked features from the existing crop only.
CROP_BIN_FACTOR = 2
CROP_POD_RANK = 24
TRAIN_SIZES = [64, 128, 256, 512]
TRAIN_SUBSAMPLE_SEEDS = [11, 23, 37]
MAX_VALIDATION_FRAMES = 512
FINAL_TRAIN_SIZE = 512  # Prespecified size, not chosen from test.
MAX_TEST_FRAMES = 256
CLASSIFIER_GRID = {
    "C_values": [0.1, 1.0, 10.0],
    "gamma_values": [0.1, 1.0, "scale"],
    "angle_scales": [0.1, 0.25, 0.5, 1.0],
    "reps": [1, 2],
    "entanglement": "linear",
    "class_weight": "balanced",
}
FIGURE_DPI = 600
BLOCK_LENGTH_CAMERA_FRAMES = 256  # Assess using TRAINING correlation; this is a starting value.
N_BLOCK_BOOTSTRAPS = 500
TEST_USAGE = "previously_inspected"  # Current record has already been evaluated.
# Use "unseen_record_or_holdout" only if that is factually true for the source experiment.

PREPARE_REPEAT_SAMPLE = False
OPEN_BLIND_ROUND_1 = False
OPEN_BLIND_ROUND_2 = False
RUN_MEASUREMENT_REPORT = False
RUN_VALIDATION_STUDY = False
RUN_FROZEN_TEST = False

def latest_measurement_report():
    candidates = sorted(MEASUREMENT_REPORT_ROOT.glob("report_*/measurement_report.json"))
    return candidates[-1] if candidates else None

def display_figures(directory, patterns):
    directory = Path(directory)
    for pattern in patterns:
        for path in sorted(directory.glob(pattern)):
            display(Image(filename=str(path), width=1100))

print("Saved TP/POD experiment:", RESULTS_DIRECTORY)
print("Conference outputs:", PAPER_DIRECTORY)
print("New blind annotations:", REPEAT_DIRECTORY)
print("Fixed classifier dimension:", FEATURE_DIMENSION)

## 2. Check the source experiment and prepare blind repeat frames

The repeat sample spans training and validation times; it contains no test frames and no displayed tracked coordinates. It is sampled evenly in time, so inspect whether extremes and difficult images are represented. This is a repeatability sample, not a guarantee of full-record tracking accuracy. Sample identities are saved once and reused for both rounds.

In [ ]:
if (RESULTS_DIRECTORY / "modal_coefficients.csv").is_file() and STATES_CSV.is_file():
    source = load_study(RESULTS_DIRECTORY, STATES_CSV)
    print("Image directory:", source["config"]["image_directory"])
    print("Saved POD crop:", source["config"].get("roi"))
    print("Training/validation frames:", {p: len(v) for p,v in source["split"].items()})
    print("Training TP motion SD (not localization uncertainty):",
          source["definition"]["standard_deviation_y_px"], "px")
    if PREPARE_REPEAT_SAMPLE:
        if (REPEAT_DIRECTORY / "repeat_manifest.json").is_file():
            print("Repeat sample already saved; reuse it for both blind rounds.")
        else:
            display(prepare_repeat_sample(RESULTS_DIRECTORY, STATES_CSV, REPEAT_DIRECTORY,
                    n_train=N_REPEAT_TRAIN, n_validation=N_REPEAT_VALIDATION))
else:
    print("Complete the reviewed TP/POD workflow first, or correct RESULTS_DIRECTORY and STATES_CSV.")

## 3. Two blind marking rounds

Enable one round at a time and rerun parameters plus its cell. The desktop shows the original TIFF **without existing TP markers**. Click TP and press Enter to save; S defers, Backspace revisits, and closing retains saved points. Each round has a different shuffled order. Complete round 2 in a separate session without consulting round 1; an additional independent annotator is useful when possible.

Both rounds are stored separately from the original tracking/manual CSVs. They do not change class boundaries or automatically modify the track. If they reveal tracking errors, correct the original experiment and rerun the source category/POD study before continuing.

In [ ]:
if OPEN_BLIND_ROUND_1:
    if (REPEAT_DIRECTORY / "repeat_manifest.json").is_file():
        display(launch_repeat_annotation(REPEAT_DIRECTORY, round_number=1, backend=DESKTOP_BACKEND))
    else:
        print("Prepare the repeat sample first.")
if OPEN_BLIND_ROUND_2:
    if (REPEAT_DIRECTORY / "repeat_manifest.json").is_file():
        display(launch_repeat_annotation(REPEAT_DIRECTORY, round_number=2, backend=DESKTOP_BACKEND))
    else:
        print("Prepare the repeat sample first.")

## 4. Estimate repeatability and tracking agreement

For repeated y measurements, the single-annotation random repeatability estimate is `SD(y₂−y₁)/sqrt(2)`, assuming independent equal-variance errors. Report inter-round bias separately. The RMS difference divided by sqrt(2) also reflects inter-round bias. Neither captures common annotation bias, optical calibration errors or full ground-truth uncertainty.

The mean of the two clicks is a reference for agreement, not exact truth. Training-repeat measurements alone define the secondary boundary sensitivity width: `2 × max(repeatability SD, RMS difference / sqrt(2))`. It is **not a calibrated 95% confidence band**. Validation repeat results audit agreement and are not used to tune this width. No report is fabricated if repeat clicks are missing.

In [ ]:
if RUN_MEASUREMENT_REPORT:
    if (REPEAT_DIRECTORY / "repeat_manifest.json").is_file():
        measurement_output = MEASUREMENT_REPORT_ROOT / datetime.now().strftime("report_%Y%m%d_%H%M%S_%f")
        measurement = measurement_report(REPEAT_DIRECTORY, measurement_output, dpi=FIGURE_DPI)
        display(measurement)
        display_figures(measurement_output, ["01*.png"])
    else:
        print("Prepare the sample and complete both blind rounds first.")
else:
    report_path = latest_measurement_report()
    if report_path:
        display(json.loads(report_path.read_text()))
    else:
        print("No repeat-measurement report yet.")

## 5. Run the validation-only conference study

All representations use `FEATURE_DIMENSION` inputs: leading POD coefficients or coefficients ranked using the **current sampled training rows' TP correlations**, for each crop. Both classifiers share those rows and the same training-fitted [0,π] mapping. Constant coordinates remain frozen. Validation clipping is reported.

Subsets are nested within each seeded training permutation. Curves show mean ± SD over these training subsets—not independent experiments or confidence intervals. POD uses all unlabeled training images, and class boundaries use all training TP measurements. Therefore these are **classifier sample-size curves, not total annotation-efficiency curves**.

The selected common representation maximizes the average of both models' mean validation balanced accuracies at `FINAL_TRAIN_SIZE`. Each model's hyperparameters are selected on validation. The first seed is frozen, never the best-performing seed. Ridge regression with alpha=1 provides a continuous-position diagnostic using the same selected columns. No test value is evaluated at this stage.

In [ ]:
if RUN_VALIDATION_STUDY:
    report_path = latest_measurement_report()
    report = json.loads(report_path.read_text()) if report_path else {"status": "pending"}
    if REQUIRE_MEASURED_UNCERTAINTY and report.get("status") != "complete":
        print("Complete both blind rounds and the uncertainty report first.")
        print("For an explicitly exploratory run only, set REQUIRE_MEASURED_UNCERTAINTY=False.")
    elif (VALIDATION_DIRECTORY / "selection.json").is_file():
        print("This validation study is already frozen. Use a new STUDY_TAG for changed settings.")
    else:
        validation_study = run_validation_study(
            RESULTS_DIRECTORY, STATES_CSV, VALIDATION_DIRECTORY,
            dimension=FEATURE_DIMENSION, crops=CROP_STUDIES,
            crop_bin_factor=CROP_BIN_FACTOR, crop_rank=CROP_POD_RANK,
            train_sizes=TRAIN_SIZES, seeds=TRAIN_SUBSAMPLE_SEEDS,
            max_validation=MAX_VALIDATION_FRAMES, final_train_size=FINAL_TRAIN_SIZE,
            classifier=CLASSIFIER_GRID, measurement_report=report_path,
            assumed_margin_px=ASSUMED_SENSITIVITY_MARGIN_PX, dpi=FIGURE_DPI,
        )
        display(validation_study)
if (VALIDATION_DIRECTORY / "selection.json").is_file():
    selection = json.loads((VALIDATION_DIRECTORY / "selection.json").read_text())
    display({k:selection[k] for k in ("representation", "dimension", "train_size", "seed", "columns", "margin_provenance")})
    print("Selected columns are zero-based above; CSV mode numbers are one-based.")
    display_figures(VALIDATION_DIRECTORY, ["02*.png", "03*.png", "04*.png", "05*.png"])

## 6. Inspect boundary errors and computational cost before test

Primary metrics include every sampled validation image. The secondary boundary report excludes only a prespecified/measurement-derived band and reports retained counts and coverage. Bin counts accompany error rates; no points are silently discarded to improve the primary result.

Cost CSVs separate hyperparameter-search wall time from selected-model prediction time. New crop costs include preprocessing, training POD and projection. Historical costs for the reused POD are marked unmeasured. Quantum timings are **classical exact-statevector simulator costs**, not quantum hardware runtime or evidence of computational speedup.

In [ ]:
if (VALIDATION_DIRECTORY / "selection.json").is_file():
    display_figures(VALIDATION_DIRECTORY, ["validation_boundary_errors.png", "06*.png"])
    display(json.loads((VALIDATION_DIRECTORY / "validation_boundary_sensitivity.json").read_text()))
    display(json.loads((VALIDATION_DIRECTORY / "feature_sources.json").read_text()))
    print("Detailed tables:", VALIDATION_DIRECTORY)
else:
    print("Run the validation study first.")

## 7. Separately evaluate the frozen test recipe

Enable `RUN_FROZEN_TEST` only after freezing the planned study. This reconstructs both selected models from saved training angles and hyperparameters; it performs no validation/test search or validation refit. It evaluates the same sampled test frames for both models. The test folder is protected from rerunning over existing outputs.

The existing record has already been inspected, so `TEST_USAGE="previously_inspected"` is appropriate. Do not relabel it as unseen. A new final generalization claim requires an actually uninspected holdout/record processed under the frozen protocol.

Paired block-bootstrap differences are a sensitivity analysis to temporal dependence. Choose block length from training correlation and assess sensitivity. Too few blocks/class-complete draws produce an explicit insufficient-data status, not a spurious interval.

In [ ]:
if RUN_FROZEN_TEST:
    if not (VALIDATION_DIRECTORY / "selection.json").is_file():
        print("Freeze validation selection first.")
    elif (FINAL_DIRECTORY / "final_evaluation.json").is_file():
        print("Frozen test already evaluated. Read its outputs; do not tune on them.")
    else:
        final = evaluate_frozen_test(
            VALIDATION_DIRECTORY, FINAL_DIRECTORY, max_test=MAX_TEST_FRAMES,
            test_usage=TEST_USAGE, block_frames=BLOCK_LENGTH_CAMERA_FRAMES,
            n_bootstrap=N_BLOCK_BOOTSTRAPS, dpi=FIGURE_DPI,
        )
        display({k:final[k] for k in ("test_usage", "representation", "n_test", "metrics", "timing_seconds")})
if (FINAL_DIRECTORY / "final_evaluation.json").is_file():
    final = json.loads((FINAL_DIRECTORY / "final_evaluation.json").read_text())
    display(final["paired_temporal_block_difference"])
    display(final["secondary_boundary_sensitivity"])
    display_figures(FINAL_DIRECTORY, ["07*.png", "test_boundary_errors.png"])

## 8. Export the analysis-status checklist

This records what has actually been completed. It is not an automatic scientific sign-off. Keep your experimental methods, calibration, actual exposure/frame timing, tracking review counts, state support and independent-record limitations explicit in the paper. Missing/poor measurements cannot be repaired by a classifier comparison.

In [ ]:
report_path = latest_measurement_report()
measurement = json.loads(report_path.read_text()) if report_path else {"status":"pending"}
final_path = FINAL_DIRECTORY / "final_evaluation.json"
final = json.loads(final_path.read_text()) if final_path.is_file() else {}
checks = {
    "repeat_measurement_status": measurement.get("status"),
    "validation_selection_frozen": (VALIDATION_DIRECTORY / "selection.json").is_file(),
    "test_evaluated": bool(final),
    "test_usage": final.get("test_usage", "not_evaluated"),
    "fresh_final_test_evidence": final.get("test_usage") == "unseen_record_or_holdout",
    "all_test_classes_supported": all(v>0 for v in final.get("metrics",{}).get("rbf_svm",{}).get("class_support",{}).values()) if final else False,
    "experimental_TP_accuracy_and_calibration": "Requires experimental assessment; repeatability is not total uncertainty.",
    "paper_completeness": "Requires author review; software execution alone does not complete scientific validation.",
}
if PAPER_DIRECTORY.exists(): write_json(PAPER_DIRECTORY / "analysis_status.json", checks)
display(checks)

## Output guide

- **repeat_annotations/**: sampled image identities, two blind point CSVs, source hashes.
- **measurement_reports/**: localization repeatability, tracking-versus-repeat agreement, inter-round bias, category disagreement and figures.
- **validation_study/**: mode correlations, continuous-position regression, matched learning curves, full search and fixed-dimension comparison tables, boundary errors, wall times, crop PODs, and frozen recipe.
- **frozen_test/**: actual paired test predictions, complete class-support metrics, count-annotated confusion matrices, boundary coverage/error tables, paired temporal-block sensitivity and selected-model computational cost.

All figures are PNG/PDF/SVG with inspectable numeric CSV/JSON sources. The workflow does not promise QSVM improvement or quantum advantage. It provides a reproducible basis for reporting whichever outcome the measurements support.